In [3]:
import pandas as pd
from mlxtend.frequent_patterns import fpgrowth, association_rules

lines = pd.read_parquet("../data/marts/basket_lines.parquet")
names = lines.drop_duplicates("stock_code").set_index("stock_code")["product_name"]
lines.shape, lines["invoice"].nunique(), lines["stock_code"].nunique()

((992051, 4), 39516, 4724)

In [9]:
def basket_matrix(df, min_item_support=0.01):
    """เก็บเฉพาะสินค้าที่ support ถึงเกณฑ์ แต่เก็บทุกใบเสร็จไว้เป็นตัวหาร"""
    all_invoices = df["invoice"].unique()
    item_support = df.groupby("stock_code")["invoice"].nunique() / len(all_invoices)
    keep = item_support[item_support >= min_item_support].index
    kept = df[df["stock_code"].isin(keep)]
    m = (pd.crosstab(kept["invoice"], kept["stock_code"])
            .gt(0)
            .reindex(all_invoices, fill_value=False))    # ← สำคัญ: ดูกับดักด้านล่าง
    return m

retail = basket_matrix(lines[lines["customer_type"] == "retail"])
retail.shape

(33241, 554)

In [12]:
freq = fpgrowth(retail, min_support=0.01, use_colnames=True, max_len=3)
rules = association_rules(freq, num_itemsets=len(retail), metric="lift", min_threshold=1.0)
rules = rules[rules["confidence"] >= 0.3].sort_values("lift", ascending=False)
len(freq), len(rules)

(848, 489)

In [14]:
def label(items):
    return " + ".join(names[i] for i in sorted(items))

view = rules.assign(if_buy=rules["antecedents"].map(label),
                    also_buy=rules["consequents"].map(label))
view[["if_buy", "also_buy", "support", "confidence", "lift"]].head(15).round(3)

,if_buy,also_buy,support,confidence,lift
277,POPPY'S PLAYHOUSE LIVINGROOM,POPPY'S PLAYHOUSE BEDROOM,0.010,0.823,54.522
276,POPPY'S PLAYHOUSE BEDROOM,POPPY'S PLAYHOUSE LIVINGROOM,0.010,0.687,54.522
275,POPPY'S PLAYHOUSE LIVINGROOM,POPPY'S PLAYHOUSE KITCHEN,0.011,0.883,52.324
274,POPPY'S PLAYHOUSE KITCHEN,POPPY'S PLAYHOUSE LIVINGROOM,0.011,0.660,52.324
273,POPPY'S PLAYHOUSE KITCHEN,POPPY'S PLAYHOUSE BEDROOM,0.013,0.759,50.283
272,POPPY'S PLAYHOUSE BEDROOM,POPPY'S PLAYHOUSE KITCHEN,0.013,0.849,50.283
516,SET/6 RED SPOTTY PAPER CUPS,SET/6 RED SPOTTY PAPER PLATES,0.012,0.796,46.407
517,SET/6 RED SPOTTY PAPER PLATES,SET/6 RED SPOTTY PAPER CUPS,0.012,0.725,46.407
498,WOODEN TREE CHRISTMAS SCANDINAVIAN,WOODEN STAR CHRISTMAS SCANDINAVIAN,0.010,0.790,42.748
499,WOODEN STAR CHRISTMAS SCANDINAVIAN,WOODEN TREE CHRISTMAS SCANDINAVIAN,0.010,0.544,42.748


In [15]:
rules["lift"].describe().round(2)

count    489.00
mean      14.45
std        9.72
min        2.17
25%        8.61
50%       10.60
75%       16.52
max       54.52
Name: lift, dtype: float64

In [16]:
import numpy as np

pairs = rules[(rules["antecedents"].map(len) == 1) & (rules["consequents"].map(len) == 1)].copy()
pairs["a"] = pairs["antecedents"].map(lambda s: next(iter(s)))
pairs["b"] = pairs["consequents"].map(lambda s: next(iter(s)))

def name_overlap(a, b):
    """สัดส่วนคำที่ซ้ำกันในชื่อสินค้า 2 ตัว (Jaccard similarity)"""
    x, y = set(names[a].split()), set(names[b].split())
    return len(x & y) / len(x | y)

pairs["name_overlap"] = [name_overlap(a, b) for a, b in zip(pairs["a"], pairs["b"])]
pairs["rule_type"] = np.where(pairs["name_overlap"] >= 0.5, "complete the set", "cross-sell")
pairs["rule_type"].value_counts()

rule_type
cross-sell          204
complete the set    138
Name: count, dtype: int64

In [17]:
def top_rules(p, rule_type, n=10):
    out = p[p["rule_type"] == rule_type].sort_values("lift", ascending=False).head(n)
    return out.assign(if_buy=out["a"].map(names), also_buy=out["b"].map(names))[
        ["if_buy", "also_buy", "support", "confidence", "lift"]].round(3)

top_rules(pairs, "cross-sell")

,if_buy,also_buy,support,confidence,lift
208,SET/10 PINK POLKADOT PARTY CANDLES,SET/10 BLUE SPOTTY PARTY CANDLES,0.013,0.648,41.082
209,SET/10 BLUE SPOTTY PARTY CANDLES,SET/10 PINK POLKADOT PARTY CANDLES,0.013,0.800,41.082
771,GARDENERS KNEELING PAD KEEP CALM,GARDENERS KNEELING PAD CUP OF TEA,0.013,0.611,34.647
770,GARDENERS KNEELING PAD CUP OF TEA,GARDENERS KNEELING PAD KEEP CALM,0.013,0.715,34.647
238,BLACK/BLUE POLKADOT UMBRELLA,RED RETROSPOT UMBRELLA,0.010,0.607,29.162
239,RED RETROSPOT UMBRELLA,BLACK/BLUE POLKADOT UMBRELLA,0.010,0.496,29.162
665,RED RETROSPOT SUGAR JAM BOWL,RED RETROSPOT SMALL MILK JUG,0.011,0.658,25.387
664,RED RETROSPOT SMALL MILK JUG,RED RETROSPOT SUGAR JAM BOWL,0.011,0.419,25.387
455,DOLLY GIRL LUNCH BOX,SPACEBOY LUNCH BOX,0.016,0.659,23.955
454,SPACEBOY LUNCH BOX,DOLLY GIRL LUNCH BOX,0.016,0.595,23.955


In [18]:
def mine(df, min_support=0.01, min_conf=0.3):
    m = basket_matrix(df, min_item_support=min_support)
    f = fpgrowth(m, min_support=min_support, use_colnames=True, max_len=3)
    r = association_rules(f, num_itemsets=len(m), metric="lift", min_threshold=1.0)
    return m, r[r["confidence"] >= min_conf]

wholesale, rules_w = mine(lines[lines["customer_type"] == "wholesale"])
wholesale.shape, len(rules_w)

((3353, 711), 1743)

In [19]:
def same_product_type(a, b, n=2):
    """คำท้าย n คำเหมือนกัน → สินค้าประเภทเดียวกัน ต่างแค่สีหรือลาย"""
    return names[a].split()[-n:] == names[b].split()[-n:]

def classify(p):
    p = p.copy()
    p["name_overlap"] = [name_overlap(a, b) for a, b in zip(p["a"], p["b"])]
    p["same_type"]    = [same_product_type(a, b) for a, b in zip(p["a"], p["b"])]
    p["rule_type"] = np.where((p["name_overlap"] >= 0.5) | p["same_type"],
                              "complete the set", "cross-sell")
    return p

pairs = classify(pairs)
pairs["rule_type"].value_counts()

rule_type
complete the set    178
cross-sell          164
Name: count, dtype: int64

In [20]:
top_rules(pairs, "cross-sell", n=20)

,if_buy,also_buy,support,confidence,lift
771,GARDENERS KNEELING PAD KEEP CALM,GARDENERS KNEELING PAD CUP OF TEA,0.013,0.611,34.647
770,GARDENERS KNEELING PAD CUP OF TEA,GARDENERS KNEELING PAD KEEP CALM,0.013,0.715,34.647
238,BLACK/BLUE POLKADOT UMBRELLA,RED RETROSPOT UMBRELLA,0.010,0.607,29.162
239,RED RETROSPOT UMBRELLA,BLACK/BLUE POLKADOT UMBRELLA,0.010,0.496,29.162
665,RED RETROSPOT SUGAR JAM BOWL,RED RETROSPOT SMALL MILK JUG,0.011,0.658,25.387
664,RED RETROSPOT SMALL MILK JUG,RED RETROSPOT SUGAR JAM BOWL,0.011,0.419,25.387
644,RECYCLING BAG RETROSPOT,TOY TIDY PINK POLKADOT,0.011,0.419,19.384
645,TOY TIDY PINK POLKADOT,RECYCLING BAG RETROSPOT,0.011,0.531,19.384
551,HAND WARMER OWL DESIGN,HAND WARMER RED RETROSPOT,0.011,0.350,17.126
550,HAND WARMER RED RETROSPOT,HAND WARMER OWL DESIGN,0.011,0.515,17.126


In [21]:
DESCRIPTORS = {
    # สี
    "RED", "PINK", "BLUE", "GREEN", "WHITE", "BLACK", "IVORY", "CREAM", "YELLOW",
    "PURPLE", "ORANGE", "BROWN", "GREY", "GOLD", "SILVER", "PASTEL", "NATURAL",
    # ลาย / ธีม
    "SPOT", "SPOTS", "SPOTTY", "POLKADOT", "RETROSPOT", "STRIPE", "STRIPES", "STRIPEY",
    "FLORAL", "VINTAGE", "PAISLEY", "SKULL", "SKULLS", "SPACEBOY", "DOLLY", "GIRL",
    "HEART", "HEARTS", "LEAF", "BIRD", "CHARLOTTE", "REGENCY", "ROSES",
    # คำทั่วไป
    "SET", "OF", "SET/10", "SET/6", "SET/4", "SET/3", "SET/2", "AND", "&", "WITH",
}

def core_words(code):
    """ชื่อสินค้าที่ตัดสี ลาย และคำทั่วไปออกแล้ว เหลือแค่ประเภทสินค้า"""
    return set(names[code].split()) - DESCRIPTORS

def same_product_type(a, b):
    x, y = core_words(a), core_words(b)
    if not x or not y:
        return False
    return len(x & y) / len(x | y) >= 0.5

pairs = classify(pairs)
pairs["rule_type"].value_counts()

rule_type
complete the set    230
cross-sell          112
Name: count, dtype: int64

In [22]:
top_rules(pairs, "cross-sell", n=20)

,if_buy,also_buy,support,confidence,lift
771,GARDENERS KNEELING PAD KEEP CALM,GARDENERS KNEELING PAD CUP OF TEA,0.013,0.611,34.647
770,GARDENERS KNEELING PAD CUP OF TEA,GARDENERS KNEELING PAD KEEP CALM,0.013,0.715,34.647
665,RED RETROSPOT SUGAR JAM BOWL,RED RETROSPOT SMALL MILK JUG,0.011,0.658,25.387
664,RED RETROSPOT SMALL MILK JUG,RED RETROSPOT SUGAR JAM BOWL,0.011,0.419,25.387
710,RED RETROSPOT SMALL MILK JUG,RETRO SPOT LARGE MILK JUG,0.012,0.459,19.677
711,RETRO SPOT LARGE MILK JUG,RED RETROSPOT SMALL MILK JUG,0.012,0.510,19.677
644,RECYCLING BAG RETROSPOT,TOY TIDY PINK POLKADOT,0.011,0.419,19.384
645,TOY TIDY PINK POLKADOT,RECYCLING BAG RETROSPOT,0.011,0.531,19.384
513,SET OF 20 VINTAGE CHRISTMAS NAPKINS,60 CAKE CASES VINTAGE CHRISTMAS,0.010,0.408,17.030
512,60 CAKE CASES VINTAGE CHRISTMAS,SET OF 20 VINTAGE CHRISTMAS NAPKINS,0.010,0.419,17.030


In [23]:
def to_pairs(r):
    p = r[(r["antecedents"].map(len) == 1) & (r["consequents"].map(len) == 1)].copy()
    p["a"] = p["antecedents"].map(lambda s: next(iter(s)))
    p["b"] = p["consequents"].map(lambda s: next(iter(s)))
    return classify(p)

pairs_w = to_pairs(rules_w)

summary = pd.concat([
    pairs.assign(segment="retail"),
    pairs_w.assign(segment="wholesale"),
]).assign(if_buy=lambda d: d["a"].map(names), also_buy=lambda d: d["b"].map(names))

cols = ["segment", "rule_type", "if_buy", "also_buy", "support", "confidence", "lift"]
(summary.sort_values("lift", ascending=False)
        .groupby(["segment", "rule_type"]).head(10)[cols]
        .to_csv("../reports/basket_top_rules.csv", index=False))

summary.groupby(["segment", "rule_type"]).size()

segment    rule_type       
retail     complete the set    230
           cross-sell          112
wholesale  complete the set    444
           cross-sell          411
dtype: int64

In [24]:
DESCRIPTORS = {
    # สี
    "RED", "PINK", "BLUE", "GREEN", "WHITE", "BLACK", "IVORY", "CREAM", "YELLOW",
    "PURPLE", "ORANGE", "BROWN", "GREY", "GOLD", "SILVER", "PASTEL", "NATURAL",
    # ลาย / ธีม
    "SPOT", "SPOTS", "SPOTTY", "POLKADOT", "RETROSPOT", "STRIPE", "STRIPES", "STRIPEY",
    "FLORAL", "VINTAGE", "PAISLEY", "SKULL", "SKULLS", "SPACEBOY", "DOLLY", "GIRL",
    "HEART", "HEARTS", "LEAF", "BIRD", "CHARLOTTE", "REGENCY", "ROSES",
    # ขนาด
    "SMALL", "MEDIUM", "LARGE", "MINI", "JUMBO", "BIG", "GIANT", "TINY",
    # คำที่สะกดแยก
    "RETRO",
    # คำทั่วไป
    "SET", "OF", "SET/10", "SET/6", "SET/4", "SET/3", "SET/2", "AND", "&", "WITH",
}

def core_words(code):
    """ชื่อสินค้าที่ตัดสี ลาย และคำทั่วไปออกแล้ว เหลือแค่ประเภทสินค้า"""
    return set(names[code].split()) - DESCRIPTORS

def same_product_type(a, b):
    x, y = core_words(a), core_words(b)
    if not x or not y:
        return False
    return len(x & y) / len(x | y) >= 0.5

pairs = classify(pairs)
pairs["rule_type"].value_counts()

rule_type
complete the set    226
cross-sell          116
Name: count, dtype: int64

In [25]:
top_rules(pairs, "cross-sell", n=20)

,if_buy,also_buy,support,confidence,lift
771,GARDENERS KNEELING PAD KEEP CALM,GARDENERS KNEELING PAD CUP OF TEA,0.013,0.611,34.647
770,GARDENERS KNEELING PAD CUP OF TEA,GARDENERS KNEELING PAD KEEP CALM,0.013,0.715,34.647
665,RED RETROSPOT SUGAR JAM BOWL,RED RETROSPOT SMALL MILK JUG,0.011,0.658,25.387
664,RED RETROSPOT SMALL MILK JUG,RED RETROSPOT SUGAR JAM BOWL,0.011,0.419,25.387
644,RECYCLING BAG RETROSPOT,TOY TIDY PINK POLKADOT,0.011,0.419,19.384
645,TOY TIDY PINK POLKADOT,RECYCLING BAG RETROSPOT,0.011,0.531,19.384
513,SET OF 20 VINTAGE CHRISTMAS NAPKINS,60 CAKE CASES VINTAGE CHRISTMAS,0.010,0.408,17.030
512,60 CAKE CASES VINTAGE CHRISTMAS,SET OF 20 VINTAGE CHRISTMAS NAPKINS,0.010,0.419,17.030
733,ROUND SNACK BOXES SET OF4 WOODLAND,ROUND SNACK BOXES SET OF 4 FRUITS,0.012,0.418,16.523
732,ROUND SNACK BOXES SET OF 4 FRUITS,ROUND SNACK BOXES SET OF4 WOODLAND,0.012,0.485,16.523


In [26]:
def to_pairs(r):
    p = r[(r["antecedents"].map(len) == 1) & (r["consequents"].map(len) == 1)].copy()
    p["a"] = p["antecedents"].map(lambda s: next(iter(s)))
    p["b"] = p["consequents"].map(lambda s: next(iter(s)))
    return classify(p)

pairs_w = to_pairs(rules_w)

summary = pd.concat([
    pairs.assign(segment="retail"),
    pairs_w.assign(segment="wholesale"),
]).assign(if_buy=lambda d: d["a"].map(names), also_buy=lambda d: d["b"].map(names))

cols = ["segment", "rule_type", "if_buy", "also_buy", "support", "confidence", "lift"]
(summary.sort_values("lift", ascending=False)
        .groupby(["segment", "rule_type"]).head(10)[cols]
        .to_csv("../reports/basket_top_rules.csv", index=False))

summary.groupby(["segment", "rule_type"]).size()

segment    rule_type       
retail     complete the set    226
           cross-sell          116
wholesale  complete the set    438
           cross-sell          417
dtype: int64